# Week 77, Day 6: Forge

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w077d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('miniflask.py', 'w', encoding='utf-8') as f:
    f.write('"""A small subset of Flask that runs in the page (Week 77).\n\nThe names and behaviour follow Flask\'s own, so code written against this module also runs with\nthe real Flask (pip install flask): Flask, request, g, jsonify, abort, redirect, make_response,\nurl_for, Blueprint, and app.test_client(). It\'s a teaching copy, not the real thing: no templates,\nsessions, streaming or debugger.\n"""\nimport io\nimport json\nimport re\nfrom contextvars import ContextVar\nfrom http import HTTPStatus\nfrom urllib.parse import parse_qs, urlencode, urlsplit\n\n_current = ContextVar("request")\n\n\nclass HTTPException(Exception):\n    def __init__(self, code, description=None):\n        self.code = code\n        self.name = HTTPStatus(code).phrase\n        self.description = description or self.name\n        super().__init__(f"{code} {self.name}: {self.description}")\n\n\ndef abort(code, description=None):\n    raise HTTPException(code, description)\n\n\nclass Headers:\n    """Case-insensitive headers."""\n    def __init__(self, items=()):\n        self._items = [(k, str(v)) for k, v in (items.items() if isinstance(items, dict) else items)]\n\n    def get(self, name, default=None):\n        for k, v in self._items:\n            if k.lower() == name.lower():\n                return v\n        return default\n\n    def __getitem__(self, name):\n        value = self.get(name)\n        if value is None:\n            raise KeyError(name)\n        return value\n\n    def __contains__(self, name):\n        return self.get(name) is not None\n\n    def __setitem__(self, name, value):\n        self._items = [(k, v) for k, v in self._items if k.lower() != name.lower()] + [(name, str(value))]\n\n    def items(self):\n        return list(self._items)\n\n\nclass Args(dict):\n    """Query or form values: .get(name) gives the first value; .getlist(name) all of them."""\n    def __init__(self, pairs):\n        super().__init__()\n        self._lists = {}\n        for k, v in pairs:\n            self._lists.setdefault(k, []).append(v)\n            dict.setdefault(self, k, v)\n\n    def get(self, key, default=None, type=None):\n        if key not in self:\n            return default\n        value = self[key]\n        if type is not None:\n            try:\n                return type(value)\n            except (TypeError, ValueError):\n                return default\n        return value\n\n    def getlist(self, key):\n        return list(self._lists.get(key, []))\n\n\nclass Request:\n    def __init__(self, environ):\n        self.environ = environ\n        self.method = environ["REQUEST_METHOD"].upper()\n        self.path = environ.get("PATH_INFO", "/") or "/"\n        self.args = Args(_pairs(environ.get("QUERY_STRING", "")))\n        items = [(k[5:].replace("_", "-").title(), v) for k, v in environ.items() if k.startswith("HTTP_")]\n        if environ.get("CONTENT_TYPE"):\n            items.append(("Content-Type", environ["CONTENT_TYPE"]))\n        self.headers = Headers(items)\n        length = int(environ.get("CONTENT_LENGTH") or 0)\n        self.data = environ["wsgi.input"].read(length) if length else b""\n        self.content_type = environ.get("CONTENT_TYPE", "")\n        self.form = Args(_pairs(self.data.decode("utf-8"))) if self.content_type.startswith("application/x-www-form-urlencoded") else Args([])\n        self.view_args = {}\n        cookie = environ.get("HTTP_COOKIE", "")\n        self.cookies = dict(p.strip().split("=", 1) for p in cookie.split(";") if "=" in p)\n\n    @property\n    def is_json(self):\n        return self.content_type.split(";")[0].strip() == "application/json"\n\n    def get_json(self, silent=False, force=False):\n        if not (self.is_json or force):\n            if silent:\n                return None\n            abort(415, "expected a JSON body (Content-Type: application/json)")\n        try:\n            return json.loads(self.data or b"null")\n        except ValueError:\n            if silent:\n                return None\n            abort(400, "the body isn\'t valid JSON")\n\n    @property\n    def json(self):\n        return self.get_json()\n\n\ndef _pairs(text):\n    return [(k, v) for k, vs in parse_qs(text, keep_blank_values=True).items() for v in vs]\n\n\nclass _Proxy:\n    def __init__(self, getter):\n        object.__setattr__(self, "_getter", getter)\n\n    def __getattr__(self, name):\n        return getattr(self._getter(), name)\n\n    def __setattr__(self, name, value):\n        setattr(self._getter(), name, value)\n\n\ndef _request():\n    try:\n        return _current.get()[0]\n    except LookupError:\n        raise RuntimeError("working outside of request context") from None\n\n\ndef _g():\n    try:\n        return _current.get()[1]\n    except LookupError:\n        raise RuntimeError("working outside of request context") from None\n\n\nclass _G:\n    def get(self, name, default=None):\n        return self.__dict__.get(name, default)\n\n\nrequest = _Proxy(_request)\ng = _Proxy(_g)\n\n\nclass Response:\n    def __init__(self, response=b"", status=200, headers=None, mimetype=None, content_type=None):\n        if isinstance(response, str):\n            response = response.encode("utf-8")\n        self.data = response\n        self.status_code = status\n        self.headers = Headers(headers or {})\n        if content_type or mimetype or "Content-Type" not in self.headers:\n            ctype = content_type or (f"{mimetype}; charset=utf-8" if mimetype and mimetype.startswith("text/") else mimetype) or "text/html; charset=utf-8"\n            self.headers["Content-Type"] = ctype\n\n    @property\n    def status(self):\n        return f"{self.status_code} {HTTPStatus(self.status_code).phrase}"\n\n    @property\n    def text(self):\n        return self.data.decode("utf-8")\n\n    def get_json(self):\n        return json.loads(self.data)\n\n    @property\n    def json(self):\n        return self.get_json()\n\n    def set_cookie(self, key, value="", max_age=None, path="/", secure=False, httponly=False, samesite=None):\n        parts = [f"{key}={value}"]\n        if max_age is not None:\n            parts.append(f"Max-Age={max_age}")\n        parts.append(f"Path={path}")\n        if secure:\n            parts.append("Secure")\n        if httponly:\n            parts.append("HttpOnly")\n        if samesite:\n            parts.append(f"SameSite={samesite}")\n        self.headers._items.append(("Set-Cookie", "; ".join(parts)))\n\n\ndef jsonify(*args, **kwargs):\n    data = args[0] if len(args) == 1 else (list(args) if args else kwargs)\n    return Response(json.dumps(data), headers={"Content-Type": "application/json"})\n\n\ndef redirect(location, code=302):\n    return Response(b"", code, {"Location": location})\n\n\ndef make_response(*args):\n    rv = args[0] if len(args) == 1 else args\n    return _App._to_response(rv)\n\n\ndef url_for(endpoint, **values):\n    return _current.get()[2].url_for(endpoint, **values)\n\n\n_CONVERTERS = {"int": (r"\\d+", int), "float": (r"\\d+(?:\\.\\d+)?", float), "path": (r".+", str), "string": (r"[^/]+", str)}\n\n\nclass _Rule:\n    def __init__(self, rule, endpoint, view, methods):\n        self.rule, self.endpoint, self.view = rule, endpoint, view\n        self.methods = {m.upper() for m in methods} | ({"HEAD"} if "GET" in {m.upper() for m in methods} else set())\n        regex, self.converters = "", {}\n        for piece in re.split(r"(<[^>]+>)", rule):\n            m = re.fullmatch(r"<(?:(\\w+):)?(\\w+)>", piece)\n            if m:\n                kind, name = m.group(1) or "string", m.group(2)\n                pattern, convert = _CONVERTERS[kind]\n                regex += f"(?P<{name}>{pattern})"\n                self.converters[name] = convert\n            else:\n                regex += re.escape(piece)\n        self.regex = re.compile(regex)\n\n    def match(self, path):\n        m = self.regex.fullmatch(path)\n        if not m:\n            return None\n        return {k: self.converters[k](v) for k, v in m.groupdict().items()}\n\n    def build(self, values):\n        def fill(m):\n            return str(values.pop(m.group(2)))\n        path = re.sub(r"<(?:(\\w+):)?(\\w+)>", fill, self.rule)\n        return path + (f"?{urlencode(values)}" if values else "")\n\n\nclass _Setup:\n    def route(self, rule, methods=("GET",), endpoint=None):\n        def decorate(view):\n            self._add(rule, endpoint or view.__name__, view, methods)\n            return view\n        return decorate\n\n    def get(self, rule, **kw):\n        return self.route(rule, methods=["GET"], **kw)\n\n    def post(self, rule, **kw):\n        return self.route(rule, methods=["POST"], **kw)\n\n    def put(self, rule, **kw):\n        return self.route(rule, methods=["PUT"], **kw)\n\n    def patch(self, rule, **kw):\n        return self.route(rule, methods=["PATCH"], **kw)\n\n    def delete(self, rule, **kw):\n        return self.route(rule, methods=["DELETE"], **kw)\n\n\nclass Blueprint(_Setup):\n    def __init__(self, name, import_name, url_prefix=""):\n        self.name, self.url_prefix = name, url_prefix\n        self.deferred = []\n\n    def _add(self, rule, endpoint, view, methods):\n        self.deferred.append((rule, endpoint, view, methods))\n\n\nclass _App(_Setup):\n    def __init__(self, import_name):\n        self.import_name = import_name\n        self.config = {"TESTING": False}\n        self.rules = []\n        self.before_funcs, self.after_funcs, self.error_handlers = [], [], {}\n\n    def _add(self, rule, endpoint, view, methods):\n        self.rules.append(_Rule(rule, endpoint, view, methods))\n\n    def register_blueprint(self, bp, url_prefix=None):\n        prefix = url_prefix if url_prefix is not None else bp.url_prefix\n        for rule, endpoint, view, methods in bp.deferred:\n            self._add(prefix + rule, f"{bp.name}.{endpoint}", view, methods)\n\n    def before_request(self, f):\n        self.before_funcs.append(f)\n        return f\n\n    def after_request(self, f):\n        self.after_funcs.append(f)\n        return f\n\n    def errorhandler(self, code):\n        def decorate(f):\n            self.error_handlers[code] = f\n            return f\n        return decorate\n\n    def url_for(self, endpoint, **values):\n        for r in self.rules:\n            if r.endpoint == endpoint:\n                return r.build(dict(values))\n        raise LookupError(f"no endpoint {endpoint!r}")\n\n    @staticmethod\n    def _to_response(rv):\n        status, headers = None, None\n        if isinstance(rv, tuple):\n            if len(rv) == 3:\n                rv, status, headers = rv\n            elif isinstance(rv[1], (dict, list)):\n                rv, headers = rv\n            else:\n                rv, status = rv\n        if isinstance(rv, Response):\n            response = rv\n        elif isinstance(rv, (dict, list)):\n            response = jsonify(rv)\n        elif isinstance(rv, (str, bytes)):\n            response = Response(rv)\n        elif rv is None:\n            raise TypeError("the view function did not return a valid response")\n        else:\n            raise TypeError(f"can\'t turn {type(rv).__name__} into a response")\n        if status is not None:\n            response.status_code = status\n        for k, v in (headers.items() if isinstance(headers, dict) else headers or []):\n            response.headers[k] = v\n        return response\n\n    def _error(self, e):\n        handler = self.error_handlers.get(e.code)\n        if handler:\n            return self._to_response(handler(e))\n        body = f"<h1>{e.code} {e.name}</h1><p>{e.description}</p>"\n        return Response(body, e.code)\n\n    def _dispatch(self, req):\n        path = req.path\n        allowed = set()\n        for r in self.rules:\n            values = r.match(path)\n            if values is None:\n                continue\n            if req.method in r.methods:\n                req.view_args = values\n                return r.view(**values)\n            allowed |= r.methods\n        if allowed:\n            raise _MethodNotAllowed(sorted(allowed | {"OPTIONS"}))\n        abort(404, "The requested URL was not found on the server.")\n\n    def full_dispatch(self, environ):\n        req = Request(environ)\n        token = _current.set((req, _G(), self))\n        try:\n            try:\n                rv = None\n                for f in self.before_funcs:\n                    rv = f()\n                    if rv is not None:\n                        break\n                if rv is None:\n                    rv = self._dispatch(req)\n                response = self._to_response(rv)\n            except _MethodNotAllowed as e:\n                response = self._error(HTTPException(405))\n                response.headers["Allow"] = ", ".join(e.allowed)\n            except HTTPException as e:\n                response = self._error(e)\n            except Exception as e:\n                if self.config.get("TESTING") and 500 not in self.error_handlers:\n                    raise\n                response = self._error(HTTPException(500)) if 500 not in self.error_handlers else self._to_response(self.error_handlers[500](e))\n                response.status_code = 500\n            for f in self.after_funcs:\n                response = f(response) or response\n            if req.method == "HEAD":\n                response.headers["Content-Length"] = len(response.data)\n                response.data = b""\n            return response\n        finally:\n            _current.reset(token)\n\n    def __call__(self, environ, start_response):\n        response = self.full_dispatch(environ)\n        if "Content-Length" not in response.headers:\n            response.headers["Content-Length"] = len(response.data)\n        start_response(response.status, response.headers.items())\n        return [response.data]\n\n    def test_client(self):\n        return _TestClient(self)\n\n\nclass _MethodNotAllowed(Exception):\n    def __init__(self, allowed):\n        self.allowed = allowed\n\n\nclass Flask(_App):\n    pass\n\n\nclass _TestClient:\n    def __init__(self, app):\n        self.app = app\n        self.cookies = {}\n\n    def open(self, path, method="GET", json=None, data=None, headers=None, query_string=None):\n        url = urlsplit(path)\n        query = url.query\n        if query_string:\n            query = (query + "&" if query else "") + urlencode(query_string, doseq=True)\n        body, ctype = b"", None\n        if json is not None:\n            import json as _json\n            body, ctype = _json.dumps(json).encode("utf-8"), "application/json"\n        elif isinstance(data, dict):\n            body, ctype = urlencode(data, doseq=True).encode("utf-8"), "application/x-www-form-urlencoded"\n        elif data is not None:\n            body = data.encode("utf-8") if isinstance(data, str) else data\n        environ = {"REQUEST_METHOD": method.upper(), "PATH_INFO": url.path or "/", "QUERY_STRING": query,\n                   "SERVER_NAME": "localhost", "SERVER_PORT": "80", "wsgi.url_scheme": "http",\n                   "wsgi.input": io.BytesIO(body), "CONTENT_LENGTH": str(len(body))}\n        headers = dict(headers or {})\n        if ctype and not any(k.lower() == "content-type" for k in headers):\n            environ["CONTENT_TYPE"] = ctype\n        for k, v in headers.items():\n            if k.lower() == "content-type":\n                environ["CONTENT_TYPE"] = v\n            else:\n                environ["HTTP_" + k.upper().replace("-", "_")] = v\n        if self.cookies:\n            environ["HTTP_COOKIE"] = "; ".join(f"{k}={v}" for k, v in self.cookies.items())\n        response = self.app.full_dispatch(environ)\n        for k, v in response.headers.items():\n            if k.lower() == "set-cookie":\n                name, _, rest = v.partition("=")\n                self.cookies[name] = rest.split(";")[0]\n        return response\n\n    def get(self, path, **kw):\n        return self.open(path, "GET", **kw)\n\n    def post(self, path, **kw):\n        return self.open(path, "POST", **kw)\n\n    def put(self, path, **kw):\n        return self.open(path, "PUT", **kw)\n\n    def patch(self, path, **kw):\n        return self.open(path, "PATCH", **kw)\n\n    def delete(self, path, **kw):\n        return self.open(path, "DELETE", **kw)\n')
with open('minifast.py', 'w', encoding='utf-8') as f:
    f.write('"""A small subset of FastAPI (and of Pydantic\'s BaseModel) that runs in the page (Week 77).\n\nNames and behaviour follow FastAPI\'s own, so code written against this module also runs with the\nreal FastAPI (pip install "fastapi[standard]"): FastAPI, BaseModel, Field, Query, Header, Depends,\nHTTPException, JSONResponse, Response and TestClient (from fastapi.testclient on the real thing).\nParameters are read from your type hints: path parameters, query parameters, a JSON body from a\nBaseModel, and dependencies. Bad input gets FastAPI\'s 422 response. It\'s a teaching copy: no\nasync, no OpenAPI page, and only common types (int, float, str, bool, lists, optional, models).\n"""\nimport copy\nimport inspect\nimport json\nimport re\nimport typing\nfrom http import HTTPStatus\nfrom urllib.parse import parse_qsl, urlencode, urlsplit\n\n_MISSING = object()\n\n\n# ---------- validation -------------------------------------------------------------------------\n\nclass ValidationError(Exception):\n    def __init__(self, errors):\n        super().__init__(f"{len(errors)} validation error(s)")\n        self._errors = errors\n\n    def errors(self):\n        return list(self._errors)\n\n\nclass FieldInfo:\n    def __init__(self, default=_MISSING, ge=None, le=None, gt=None, lt=None, min_length=None, max_length=None, alias=None):\n        self.default, self.ge, self.le, self.gt, self.lt = default, ge, le, gt, lt\n        self.min_length, self.max_length, self.alias = min_length, max_length, alias\n\n\ndef Field(default=_MISSING, **kw):\n    return FieldInfo(default, **kw)\n\n\ndef _err(loc, msg, kind):\n    return {"loc": list(loc), "msg": msg, "type": kind}\n\n\ndef _optional(tp):\n    args = typing.get_args(tp)\n    if (typing.get_origin(tp) is typing.Union or type(tp).__name__ == "UnionType") and type(None) in args:\n        rest = [a for a in args if a is not type(None)]\n        return True, rest[0] if len(rest) == 1 else typing.Union[tuple(rest)]\n    return False, tp\n\n\ndef _convert(value, tp, loc, errors, from_text=False):\n    """Convert value to tp, appending errors; returns the value or _MISSING."""\n    optional, tp = _optional(tp)\n    if value is None:\n        if optional:\n            return None\n        errors.append(_err(loc, "Input should not be None", "none_required"))\n        return _MISSING\n    origin = typing.get_origin(tp)\n    if tp is typing.Any or tp is inspect.Parameter.empty:\n        return value\n    if isinstance(tp, type) and issubclass(tp, BaseModel):\n        if isinstance(value, tp):\n            return value\n        if not isinstance(value, dict):\n            errors.append(_err(loc, "Input should be a valid dictionary or object", "model_type"))\n            return _MISSING\n        return tp._validate(value, loc, errors)\n    if origin in (list, tuple, set):\n        if not isinstance(value, (list, tuple)):\n            errors.append(_err(loc, "Input should be a valid list", "list_type"))\n            return _MISSING\n        (item_type,) = typing.get_args(tp)[:1] or (typing.Any,)\n        out = [_convert(v, item_type, list(loc) + [i], errors, from_text) for i, v in enumerate(value)]\n        return _MISSING if any(v is _MISSING for v in out) else origin(out)\n    if tp is bool:\n        if isinstance(value, bool):\n            return value\n        if isinstance(value, str) and value.lower() in ("true", "1", "yes", "on", "false", "0", "no", "off"):\n            return value.lower() in ("true", "1", "yes", "on")\n        errors.append(_err(loc, "Input should be a valid boolean", "bool_parsing"))\n        return _MISSING\n    if tp is int:\n        if isinstance(value, int) and not isinstance(value, bool):\n            return value\n        if isinstance(value, str) and re.fullmatch(r"[+-]?\\d+", value.strip()):\n            return int(value)\n        if isinstance(value, float) and value.is_integer():\n            return int(value)\n        errors.append(_err(loc, "Input should be a valid integer", "int_parsing"))\n        return _MISSING\n    if tp is float:\n        if isinstance(value, (int, float)) and not isinstance(value, bool):\n            return float(value)\n        if isinstance(value, str):\n            try:\n                return float(value)\n            except ValueError:\n                pass\n        errors.append(_err(loc, "Input should be a valid number", "float_parsing"))\n        return _MISSING\n    if tp is str:\n        if isinstance(value, str):\n            return value\n        if from_text:\n            return str(value)\n        errors.append(_err(loc, "Input should be a valid string", "string_type"))\n        return _MISSING\n    if tp is dict or origin is dict:\n        if isinstance(value, dict):\n            return value\n        errors.append(_err(loc, "Input should be a valid dictionary", "dict_type"))\n        return _MISSING\n    return value\n\n\ndef _constraints(value, info, loc, errors):\n    if value is _MISSING or value is None or info is None:\n        return value\n    checks = [("ge", lambda v, n: v >= n, "greater than or equal to", "greater_than_equal"),\n              ("gt", lambda v, n: v > n, "greater than", "greater_than"),\n              ("le", lambda v, n: v <= n, "less than or equal to", "less_than_equal"),\n              ("lt", lambda v, n: v < n, "less than", "less_than")]\n    for attr, ok, words, kind in checks:\n        limit = getattr(info, attr)\n        if limit is not None and not ok(value, limit):\n            errors.append(_err(loc, f"Input should be {words} {limit}", kind))\n            return _MISSING\n    if info.min_length is not None and len(value) < info.min_length:\n        errors.append(_err(loc, f"Value should have at least {info.min_length} item{\'s\' if info.min_length != 1 else \'\'}"\n                           if not isinstance(value, str) else\n                           f"String should have at least {info.min_length} character{\'s\' if info.min_length != 1 else \'\'}",\n                           "string_too_short" if isinstance(value, str) else "too_short"))\n        return _MISSING\n    if info.max_length is not None and len(value) > info.max_length:\n        errors.append(_err(loc, f"String should have at most {info.max_length} characters", "string_too_long"))\n        return _MISSING\n    return value\n\n\nclass BaseModel:\n    """Declare fields as annotations, with defaults or Field(...)."""\n\n    @classmethod\n    def _fields(cls):\n        hints = typing.get_type_hints(cls)\n        fields = {}\n        for name, tp in hints.items():\n            if name.startswith("_"):\n                continue\n            raw = getattr(cls, name, _MISSING)\n            info = raw if isinstance(raw, FieldInfo) else FieldInfo(raw)\n            fields[name] = (tp, info)\n        return fields\n\n    @classmethod\n    def _validate(cls, data, loc, errors):\n        values = {}\n        before = len(errors)\n        for name, (tp, info) in cls._fields().items():\n            key = info.alias or name\n            if key in data:\n                value = _constraints(_convert(data[key], tp, list(loc) + [key], errors), info, list(loc) + [key], errors)\n            elif info.default is not _MISSING:\n                value = copy.deepcopy(info.default)\n            else:\n                errors.append(_err(list(loc) + [key], "Field required", "missing"))\n                value = _MISSING\n            values[name] = value\n        if len(errors) > before:\n            return _MISSING\n        obj = cls.__new__(cls)\n        obj.__dict__.update(values)\n        return obj\n\n    def __init__(self, **data):\n        errors = []\n        obj = type(self)._validate(data, [], errors)\n        if errors:\n            raise ValidationError(errors)\n        self.__dict__.update(obj.__dict__)\n\n    @classmethod\n    def model_validate(cls, data):\n        errors = []\n        obj = cls._validate(data, [], errors)\n        if errors:\n            raise ValidationError(errors)\n        return obj\n\n    def model_dump(self):\n        def dump(v):\n            if isinstance(v, BaseModel):\n                return v.model_dump()\n            if isinstance(v, (list, tuple, set)):\n                return [dump(x) for x in v]\n            return v\n        return {name: dump(getattr(self, name)) for name in type(self)._fields()}\n\n    def __eq__(self, other):\n        return type(other) is type(self) and self.model_dump() == other.model_dump()\n\n    def __repr__(self):\n        inside = " ".join(f"{k}={v!r}" for k, v in self.model_dump().items())\n        return f"{type(self).__name__}({inside})"\n\n\n# ---------- parameters and dependencies ---------------------------------------------------------\n\nclass Query(FieldInfo):\n    pass\n\n\nclass Header(FieldInfo):\n    pass\n\n\nclass _Depends:\n    def __init__(self, dependency):\n        self.dependency = dependency\n\n\ndef Depends(dependency):\n    return _Depends(dependency)\n\n\nclass HTTPException(Exception):\n    def __init__(self, status_code, detail=None, headers=None):\n        super().__init__(detail)\n        self.status_code = status_code\n        self.detail = detail if detail is not None else HTTPStatus(status_code).phrase\n        self.headers = headers or {}\n\n\nclass Response:\n    media_type = "text/plain; charset=utf-8"\n\n    def __init__(self, content=b"", status_code=200, headers=None, media_type=None):\n        self.body = content.encode("utf-8") if isinstance(content, str) else content\n        self.status_code = status_code\n        self.headers = {k.lower(): v for k, v in (headers or {}).items()}\n        if self.body:\n            self.headers.setdefault("content-type", media_type or self.media_type)\n\n\nclass JSONResponse(Response):\n    media_type = "application/json"\n\n    def __init__(self, content=None, status_code=200, headers=None):\n        super().__init__(json.dumps(content).encode("utf-8"), status_code, headers)\n\n\nclass Request:\n    def __init__(self, method, path, query, headers, body):\n        self.method, self.url_path, self.query_params = method, path, dict(query)\n        self.headers = {k.lower(): v for k, v in headers.items()}\n        self.body = body\n\n    def json(self):\n        return json.loads(self.body or b"null")\n\n\n# ---------- the application ---------------------------------------------------------------------\n\nclass FastAPI:\n    def __init__(self, title="FastAPI", version="0.1.0"):\n        self.title, self.version = title, version\n        self.routes = []\n        self.dependency_overrides = {}\n\n    def api_route(self, path, methods, status_code=200, response_model=None):\n        names = re.findall(r"{(\\w+)(?::path)?}", path)\n        regex = re.compile("^" + re.sub(r"{(\\w+)(:path)?}", lambda m: f"(?P<{m.group(1)}>{\'.+\' if m.group(2) else \'[^/]+\'})", path) + "$")\n\n        def decorate(func):\n            for method in methods:\n                self.routes.append((method, regex, names, func, status_code, response_model))\n            return func\n        return decorate\n\n    def get(self, path, **kw):\n        return self.api_route(path, ["GET"], **kw)\n\n    def post(self, path, **kw):\n        return self.api_route(path, ["POST"], **kw)\n\n    def put(self, path, **kw):\n        return self.api_route(path, ["PUT"], **kw)\n\n    def patch(self, path, **kw):\n        return self.api_route(path, ["PATCH"], **kw)\n\n    def delete(self, path, **kw):\n        return self.api_route(path, ["DELETE"], **kw)\n\n    def _solve(self, func, request, path_values, errors, cleanups, cache):\n        func = self.dependency_overrides.get(func, func)\n        if func in cache:\n            return cache[func]\n        hints = typing.get_type_hints(func)\n        kwargs = {}\n        for name, param in inspect.signature(func).parameters.items():\n            tp = hints.get(name, str if param.annotation is inspect.Parameter.empty else param.annotation)\n            default = param.default\n            if isinstance(default, _Depends):\n                kwargs[name] = self._solve(default.dependency, request, path_values, errors, cleanups, cache)\n            elif tp is Request:\n                kwargs[name] = request\n            elif name in path_values:\n                kwargs[name] = _constraints(_convert(path_values[name], tp, ["path", name], errors, True),\n                                            default if isinstance(default, FieldInfo) else None, ["path", name], errors)\n            elif isinstance(tp, type) and issubclass(tp, BaseModel):\n                try:\n                    data = json.loads(request.body) if request.body else _MISSING\n                except ValueError:\n                    errors.append(_err(["body"], "JSON decode error", "json_invalid"))\n                    continue\n                if data is _MISSING:\n                    if default is not inspect.Parameter.empty:\n                        kwargs[name] = default\n                    else:\n                        errors.append(_err(["body"], "Field required", "missing"))\n                    continue\n                kwargs[name] = _convert(data, tp, ["body"], errors)\n            else:\n                info = default if isinstance(default, FieldInfo) else FieldInfo(default if default is not inspect.Parameter.empty else _MISSING)\n                where = "header" if isinstance(default, Header) else "query"\n                key = info.alias or (name.replace("_", "-") if where == "header" else name)\n                if where == "header":\n                    raw = request.headers.get(key.lower(), _MISSING)\n                elif typing.get_origin(_optional(tp)[1]) is list:\n                    found = [v for k, v in request._pairs if k == key]\n                    raw = found if found else _MISSING\n                else:\n                    raw = request.query_params.get(key, _MISSING)\n                if raw is _MISSING:\n                    if info.default is _MISSING:\n                        errors.append(_err([where, key], "Field required", "missing"))\n                    else:\n                        kwargs[name] = info.default\n                    continue\n                kwargs[name] = _constraints(_convert(raw, tp, [where, key], errors, True), info, [where, key], errors)\n        if errors:\n            return _MISSING\n        result = func(**kwargs)\n        if inspect.isgenerator(result):\n            gen = result\n            result = next(gen)\n            cleanups.append(gen)\n        cache[func] = result\n        return result\n\n    def handle(self, method, path, query_pairs, headers, body):\n        request = Request(method, path, query_pairs, headers, body)\n        request._pairs = list(query_pairs)\n        allowed = []\n        for m, regex, names, func, status_code, response_model in self.routes:\n            match = regex.match(path)\n            if not match:\n                continue\n            if m != method:\n                allowed.append(m)\n                continue\n            errors, cleanups = [], []\n            try:\n                try:\n                    result = self._solve(func, request, match.groupdict(), errors, cleanups, {})\n                    if errors:\n                        return JSONResponse({"detail": errors}, 422)\n                except HTTPException as e:\n                    return JSONResponse({"detail": e.detail}, e.status_code, e.headers)\n                return self._respond(result, status_code, response_model)\n            finally:\n                for gen in reversed(cleanups):\n                    try:\n                        next(gen)\n                    except StopIteration:\n                        pass\n        if allowed:\n            return JSONResponse({"detail": "Method Not Allowed"}, 405, {"allow": ", ".join(sorted(set(allowed)))})\n        return JSONResponse({"detail": "Not Found"}, 404)\n\n    @staticmethod\n    def _respond(result, status_code, response_model):\n        if isinstance(result, Response):\n            return result\n        if response_model is not None:\n            data = result.model_dump() if isinstance(result, BaseModel) else result\n            if isinstance(data, list):\n                result = [{k: d[k] for k in response_model._fields()} if isinstance(d, dict) else d.model_dump() for d in\n                          [x.model_dump() if isinstance(x, BaseModel) else x for x in data]]\n            else:\n                result = {k: data[k] for k in response_model._fields()}\n        if status_code == 204:\n            return Response(b"", 204)\n\n        def plain(v):\n            if isinstance(v, BaseModel):\n                return v.model_dump()\n            if isinstance(v, (list, tuple)):\n                return [plain(x) for x in v]\n            if isinstance(v, dict):\n                return {k: plain(x) for k, x in v.items()}\n            return v\n        return JSONResponse(plain(result), status_code)\n\n\nclass _TestResponse:\n    def __init__(self, response):\n        self.status_code = response.status_code\n        self.headers = response.headers\n        self.content = response.body\n\n    @property\n    def text(self):\n        return self.content.decode("utf-8")\n\n    def json(self):\n        return json.loads(self.content)\n\n\nclass TestClient:\n    def __init__(self, app):\n        self.app = app\n\n    def request(self, method, url, params=None, json=None, headers=None, content=None):\n        parts = urlsplit(url)\n        pairs = parse_qsl(parts.query, keep_blank_values=True)\n        if params:\n            pairs += [(k, str(x)) for k, v in params.items() for x in (v if isinstance(v, list) else [v])]\n        body = b""\n        hdrs = {k.lower(): v for k, v in (headers or {}).items()}\n        if json is not None:\n            import json as _json\n            body = _json.dumps(json).encode("utf-8")\n            hdrs.setdefault("content-type", "application/json")\n        elif content is not None:\n            body = content.encode("utf-8") if isinstance(content, str) else content\n        return _TestResponse(self.app.handle(method.upper(), parts.path or "/", pairs, hdrs, body))\n\n    def get(self, url, **kw):\n        return self.request("GET", url, **kw)\n\n    def post(self, url, **kw):\n        return self.request("POST", url, **kw)\n\n    def put(self, url, **kw):\n        return self.request("PUT", url, **kw)\n\n    def patch(self, url, **kw):\n        return self.request("PATCH", url, **kw)\n\n    def delete(self, url, **kw):\n        return self.request("DELETE", url, **kw)\n')

**Predict before you run:** what will this print?



In [ ]:
from miniflask import Flask
app = Flask(__name__)

@app.route("/a/<int:n>", methods=["GET", "PUT"])
def a(n):
    return {"n": n}, 202

c = app.test_client()
print(c.get("/a/3").status_code, c.get("/a/3").get_json(), c.delete("/a/3").status_code, c.get("/a/x").status_code)

**Predict before you run:** what will this print?



In [ ]:
from minifast import FastAPI, Query, TestClient
app = FastAPI()

@app.get("/s")
def s(n: int = Query(1, ge=1, le=3)):
    return {"n": n}

c = TestClient(app)
print(c.get("/s?n=2").json(), c.get("/s?n=5").status_code, c.get("/s?n=two").json()["detail"][0]["type"])

**Predict before you run:** what will this print?



In [ ]:
import hashlib, hmac
a = hmac.new(b"k", b"msg", hashlib.sha256).hexdigest()
print(hmac.compare_digest(a, a), len(a))

**Predict before you run:** what will this print?



In [ ]:
env = {"DEBUG": "False", "PORT": "8000"}
print(bool(env["DEBUG"]), env["DEBUG"].lower() in ("1", "true", "yes"), int(env["PORT"]) + 1)

**Predict before you run:** what will this print?



In [ ]:
import base64, json
part = base64.urlsafe_b64encode(json.dumps({"sub": 7}).encode()).decode().rstrip("=")
print(json.loads(base64.urlsafe_b64decode(part + "=" * (-len(part) % 4))))

### Exercise: Errors in one format

Write `problem_errors(app)` for a miniflask app: register error handlers for `400`, `401`, `403`, `404`, `405`, `409`, `422` and `500` that answer problem details (`type` `about:blank`, `title` the exception's `name`, `status`, `detail` the description, or `"something went wrong"` for `500`) with `Content-Type: application/problem+json`. Return the app.

In [ ]:
import json
from miniflask import Flask, Response, abort

def problem_errors(app):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_problems():\n    \"problem details for every error\"\n    import json\n    from miniflask import Flask, abort\n    app = Flask(__name__)\n    @app.get(\"/conflict\")\n    def conflict():\n        abort(409, \"already there\")\n    @app.get(\"/boom\")\n    def boom():\n        raise RuntimeError(\"secret detail\")\n    problem_errors(app)\n    c = app.test_client()\n    r = c.get(\"/conflict\")\n    assert r.status_code == 409 and r.headers[\"Content-Type\"] == \"application/problem+json\"\n    assert r.get_json() == {\"type\": \"about:blank\", \"title\": \"Conflict\", \"status\": 409, \"detail\": \"already there\"}\n    assert c.get(\"/nope\").get_json()[\"title\"] == \"Not Found\" and c.post(\"/conflict\").get_json()[\"status\"] == 405\n    r = c.get(\"/boom\")\n    assert r.status_code == 500 and r.get_json()[\"detail\"] == \"something went wrong\" and \"secret\" not in r.text")

### Exercise: A settings model

With `minifast.BaseModel`, write `Settings` (`secret_key`: a `str` of at least 16 characters; `debug`: `bool`, default `False`; `port`: an `int` from 1 to 65535, default 8000; `allowed_hosts`: a list of `str`, default `["localhost"]`). Then `from_env(env)`: build the settings from environment variables named in upper case (`SECRET_KEY`, `DEBUG`, `PORT`, `ALLOWED_HOSTS`, the last comma-separated), raising `RuntimeError` whose message lists each problem as `NAME: MSG` (the field's name in upper case) joined by `"; "`.

In [ ]:
from minifast import BaseModel, Field, ValidationError

___

def from_env(env):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_settings():\n    \"validation with a model\"\n    s = from_env({\"SECRET_KEY\": \"x\" * 20, \"DEBUG\": \"true\", \"ALLOWED_HOSTS\": \"a.com, b.com\"})\n    assert (s.debug, s.port, s.allowed_hosts) == (True, 8000, [\"a.com\", \"b.com\"])\n    try:\n        from_env({\"SECRET_KEY\": \"short\", \"PORT\": \"99999\"})\n        assert False\n    except RuntimeError as e:\n        assert str(e) == \"SECRET_KEY: String should have at least 16 characters; PORT: Input should be less than or equal to 65535\"")

### Exercise: Forge: an events-booking API

Write `create_app(env, clock)` as described above.

In [ ]:
import base64, hashlib, hmac, json, secrets
from http import HTTPStatus
from miniflask import Flask, request, Response, url_for


def _b64(raw):
    return base64.urlsafe_b64encode(raw).decode().rstrip("=")


def _unb64(text):
    return base64.urlsafe_b64decode(text + "=" * (-len(text) % 4))


def _kdf(password, salt, iterations=200):
    u = hmac.new(password.encode(), salt + b"\x00\x00\x00\x01", hashlib.sha256).digest()
    total = int.from_bytes(u, "big")
    for _ in range(iterations - 1):
        u = hmac.new(password.encode(), u, hashlib.sha256).digest()
        total ^= int.from_bytes(u, "big")
    return total.to_bytes(32, "big").hex()


def hash_password(password):
    salt = secrets.token_bytes(16)
    return f"{salt.hex()}${_kdf(password, salt)}"


def verify_password(password, stored):
    salt, digest = stored.split("$")
    return hmac.compare_digest(_kdf(password, bytes.fromhex(salt)), digest)


def encode_jwt(payload, key):
    head = _b64(b'{"alg":"HS256","typ":"JWT"}') + "." + _b64(json.dumps(payload, separators=(",", ":")).encode())
    return head + "." + _b64(hmac.new(key, head.encode(), hashlib.sha256).digest())


def decode_jwt(token, key, now):
    """The payload of a valid, unexpired token; raises ValueError otherwise."""
    try:
        h, p, s = token.split(".")
        ok_alg = json.loads(_unb64(h)).get("alg") == "HS256"
        good = hmac.compare_digest(_b64(hmac.new(key, f"{h}.{p}".encode(), hashlib.sha256).digest()), s)
        payload = json.loads(_unb64(p))
    except Exception:
        raise ValueError("malformed") from None
    if not ok_alg or not good:
        raise ValueError("bad token")
    if now > payload.get("exp", 0):
        raise ValueError("expired")
    return payload


def problem(status, detail):
    """A problem-details response (RFC 9457)."""
    body = {"type": "about:blank", "title": HTTPStatus(status).phrase, "status": status, "detail": detail}
    return Response(json.dumps(body), status, {"Content-Type": "application/problem+json"})


def create_app(env, clock):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def setup():\n    now = [1000]\n    app = create_app({\"SECRET_KEY\": \"test-key\", \"TOKEN_MINUTES\": \"10\"}, lambda: now[0])\n    client = app.test_client()\n    tokens = {}\n    for email in (\"admin@example.com\", \"ada@example.com\", \"bob@example.com\"):\n        client.post(\"/users\", json={\"email\": email, \"password\": \"long enough\"})\n        token = client.post(\"/tokens\", json={\"email\": email, \"password\": \"long enough\"}).get_json()[\"token\"]\n        tokens[email.split(\"@\")[0]] = {\"Authorization\": f\"Bearer {token}\"}\n    return client, tokens, now\n\ndef test_settings_and_accounts():\n    \"configuration, registration and tokens\"\n    try:\n        create_app({}, lambda: 0)\n        assert False, \"SECRET_KEY is required\"\n    except RuntimeError:\n        pass\n    client, tokens, now = setup()\n    assert client.get(\"/healthz\").get_json() == {\"status\": \"ok\"}\n    r = client.post(\"/users\", json={\"email\": \"Cy@Example.com\", \"password\": \"long enough\"})\n    assert r.status_code == 201 and r.get_json() == {\"id\": 4, \"email\": \"cy@example.com\"}\n    r = client.post(\"/users\", json={\"email\": \"cy@example.com\", \"password\": \"long enough\"})\n    assert r.status_code == 409 and r.headers[\"Content-Type\"] == \"application/problem+json\" and r.get_json()[\"detail\"] == \"email already registered\"\n    assert client.post(\"/users\", json={\"email\": \"nope\", \"password\": \"x\"}).status_code == 422\n    assert client.post(\"/tokens\", json={\"email\": \"ada@example.com\", \"password\": \"wrong!!!!\"}).get_json()[\"detail\"] == \"wrong email or password\"\n\ndef test_events():\n    \"admin-only creation, listing and pagination\"\n    client, tokens, now = setup()\n    assert client.post(\"/events\", json={\"title\": \"Python night\", \"city\": \"Leeds\", \"capacity\": 2}).status_code == 401\n    assert client.post(\"/events\", json={\"title\": \"x\", \"city\": \"y\", \"capacity\": 2}, headers=tokens[\"ada\"]).status_code == 403\n    r = client.post(\"/events\", json={\"title\": \"Python night\", \"city\": \"Leeds\", \"capacity\": 2}, headers=tokens[\"admin\"])\n    assert r.status_code == 201 and r.headers[\"Location\"] == \"/events/1\" and r.get_json()[\"seats_left\"] == 2\n    for i in range(3):\n        client.post(\"/events\", json={\"title\": f\"Talk {i}\", \"city\": \"York\", \"capacity\": 50}, headers=tokens[\"admin\"])\n    assert client.post(\"/events\", json={\"title\": \"Huge\", \"city\": \"Hull\", \"capacity\": 5000}, headers=tokens[\"admin\"]).status_code == 422\n    page = client.get(\"/events?city=york&per_page=2&page=2\").get_json()\n    assert page[\"total\"] == 3 and [e[\"title\"] for e in page[\"items\"]] == [\"Talk 2\"]\n    assert client.get(\"/events?per_page=99\").status_code == 400 and client.get(\"/events/99\").get_json()[\"detail\"] == \"no event 99\"\n    assert client.get(\"/events/1\").get_json() == {\"id\": 1, \"title\": \"Python night\", \"city\": \"Leeds\", \"capacity\": 2, \"seats_left\": 2}\n\ndef test_bookings():\n    \"seats, duplicates, cancelling and expiry\"\n    client, tokens, now = setup()\n    client.post(\"/events\", json={\"title\": \"Python night\", \"city\": \"Leeds\", \"capacity\": 2}, headers=tokens[\"admin\"])\n    assert client.post(\"/events/1/bookings\").status_code == 401\n    r = client.post(\"/events/1/bookings\", headers=tokens[\"ada\"])\n    assert r.status_code == 201 and r.get_json() == {\"event_id\": 1, \"user_id\": 2}\n    assert client.post(\"/events/1/bookings\", headers=tokens[\"ada\"]).get_json()[\"detail\"] == \"already booked\"\n    client.post(\"/events/1/bookings\", headers=tokens[\"bob\"])\n    full = client.post(\"/events/1/bookings\", headers=tokens[\"admin\"])\n    assert full.status_code == 409 and full.get_json()[\"detail\"] == \"sold out\" and client.get(\"/events/1\").get_json()[\"seats_left\"] == 0\n    assert client.delete(\"/events/1/bookings/me\", headers=tokens[\"bob\"]).status_code == 204\n    assert client.delete(\"/events/1/bookings/me\", headers=tokens[\"bob\"]).status_code == 404\n    assert client.post(\"/events/1/bookings\", headers=tokens[\"admin\"]).status_code == 201\n    assert client.post(\"/events/7/bookings\", headers=tokens[\"ada\"]).status_code == 404\n    now[0] = 1000 + 601\n    assert client.post(\"/events/1/bookings\", headers=tokens[\"bob\"]).status_code == 401")

> **Where each rule lives.** Look back over your solution and sort the rules: which came from **HTTP and REST** (status codes, `Location`, `204`), which from **security** (hashing, constant-time comparison, token expiry, one login error message, admin checks), which from the **business** (capacity, one booking per person), and which from **operations** (refusing to start without a secret, a health check). A production service has all four kinds, and code review (Week 74) asks about each. Notice also which checks would need a database transaction (Week 75) once two people book the last seat at the same moment: in this in-memory version, they can't race, but with several workers they could.

### Exercise: Forge bonus: the same in FastAPI

Rebuild the events part of the API with `minifast` (FastAPI-style). Write `create_api(tokens, events)`: `tokens` maps tokens to `(user_id, is_admin)`; `events` is a dict by id (each with `id`, `title`, `city`, `capacity`). Models: `EventIn` (`title`: 1 to 100 characters; `city`: at least 1 character; `capacity`: an `int` from 1 to 1000) and `EventOut` (`id`, `title`, `city`, `capacity`, `seats_left`). Dependencies: `current_user`, from a `Authorization: Bearer TOKEN` header (`401`, detail `"log in first"`), and `pagination` (`page: int = Query(1, ge=1)`, `per_page: int = Query(10, ge=1, le=50)`). Endpoints: `GET /events` (with `city` optional and pagination; returns a list of `EventOut`), `POST /events` (admins only, `403` `"only admins can create events"`; `201`, `response_model=EventOut`), `GET /events/{event_id}` (`404` `"no event N"`) and `POST /events/{event_id}/bookings` (`201` with `{"event_id", "user_id"}`; `409` `"already booked"` or `"sold out"`). Keep bookings in a set inside `create_api`.

In [ ]:
from minifast import FastAPI, BaseModel, Field, Query, Header, Depends, HTTPException

def create_api(tokens, events):
    app = FastAPI()
    ___
    return app

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_fastapi_version():\n    \"models, constraints and dependencies\"\n    from minifast import TestClient\n    events = {1: {\"id\": 1, \"title\": \"Python night\", \"city\": \"Leeds\", \"capacity\": 1}}\n    client = TestClient(create_api({\"t-admin\": (1, True), \"t-ada\": (2, False), \"t-bob\": (3, False)}, events))\n    admin, ada, bob = ({\"Authorization\": f\"Bearer {t}\"} for t in (\"t-admin\", \"t-ada\", \"t-bob\"))\n    assert client.get(\"/events\").json() == [{\"id\": 1, \"title\": \"Python night\", \"city\": \"Leeds\", \"capacity\": 1, \"seats_left\": 1}]\n    r = client.post(\"/events\", json={\"title\": \"Talks\", \"city\": \"York\", \"capacity\": 30}, headers=admin)\n    assert r.status_code == 201 and r.json() == {\"id\": 2, \"title\": \"Talks\", \"city\": \"York\", \"capacity\": 30, \"seats_left\": 30}\n    assert client.post(\"/events\", json={\"title\": \"X\", \"city\": \"Y\", \"capacity\": 3}, headers=ada).json() == {\"detail\": \"only admins can create events\"}\n    assert client.post(\"/events\", json={\"title\": \"X\", \"city\": \"Y\", \"capacity\": 3}).status_code == 401\n    r = client.post(\"/events\", json={\"title\": \"\", \"city\": \"Y\", \"capacity\": 0}, headers=admin)\n    assert r.status_code == 422 and sorted(e[\"loc\"][1] for e in r.json()[\"detail\"]) == [\"capacity\", \"title\"]\n    assert [e[\"id\"] for e in client.get(\"/events?city=york\").json()] == [2] and client.get(\"/events?per_page=99\").status_code == 422\n    assert client.get(\"/events?page=2&per_page=1\").json()[0][\"id\"] == 2\n    assert client.post(\"/events/1/bookings\", headers=ada).json() == {\"event_id\": 1, \"user_id\": 2}\n    assert client.post(\"/events/1/bookings\", headers=ada).json()[\"detail\"] == \"already booked\"\n    assert client.post(\"/events/1/bookings\", headers=bob).json()[\"detail\"] == \"sold out\"\n    assert client.get(\"/events/1\").json()[\"seats_left\"] == 0 and client.get(\"/events/9\").json() == {\"detail\": \"no event 9\"}")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

## Mixed review

## Forge: an events-booking API

Build `create_app(env, clock)`: a miniflask app for booking places at events. The starter gives you `hash_password`, `verify_password`, `encode_jwt`, `decode_jwt` (which raises `ValueError` for any bad or expired token) and `problem(status, detail)`, a problem-details response. Every error the API itself reports uses `problem`; a `404` or `405` from routing does too.

- **Settings**: `SECRET_KEY` is required (raise `RuntimeError` at once if it's missing); `TOKEN_MINUTES` (default `15`) must be an integer (`RuntimeError` otherwise).
- `GET /healthz` returns `{"status": "ok"}`.
- `POST /users` with `{"email", "password"}`: emails lower-cased and containing `@`, passwords at least 8 characters (`422`, detail `"an email and a password of at least 8 characters are needed"`), duplicates `409` (`"email already registered"`). Returns `201` with `{"id", "email"}`. The **first** user is an admin.
- `POST /tokens` with the same fields returns `{"token": ...}` (a JWT with `sub` the user id and `exp` `clock() + TOKEN_MINUTES * 60`), or `401` (`"wrong email or password"`).
- `GET /events` (public) with optional `city` (any case), `page` (from 1) and `per_page` (1 to 50, default 10), else `400`; returns `{"items": [...], "page": P, "total": T}`, events by id.
- `POST /events` (admins only: `401` `"log in first"` without a valid token, `403` `"only admins can create events"`) with `{"title", "city", "capacity"}` (non-empty strings, and an integer from 1 to 1000, else `422`); `201` with the event and a `Location`.
- `GET /events/<id>`: the event (`404` `"no event N"`). Every event shown anywhere has `id`, `title`, `city`, `capacity` and `seats_left`.
- `POST /events/<id>/bookings` (logged in): `201` with `{"event_id", "user_id"}`; `404` for no event; `409` `"already booked"` or `"sold out"`.
- `DELETE /events/<id>/bookings/me` (logged in): `204`, or `404` `"no booking to cancel"`.

## Forge bonus: the same in FastAPI

## Weekly quiz

## Week 77 review

Tick off each one honestly. If one feels shaky, go back and redo two exercises from that day.

- I can build Flask apps with routes, request data, JSON responses, error handlers, hooks, blueprints and tests. ([Day 1](w077d1.html))
- I can declare FastAPI parameters and models, read 422 errors, and use dependencies and overrides. ([Day 2](w077d2.html))
- I can design REST resources and status codes, problem details, idempotency keys, ETags, cursors and compatible changes. ([Day 3](w077d3.html))
- I can hash passwords, sign cookies and JWTs, and enforce roles and ownership. ([Day 4](w077d4.html))
- I can configure an app from its environment, run it behind a proxy, add health checks and logs, containerise it and roll it out. ([Day 5](w077d5.html))
- I built a complete booking API, and part of it again in FastAPI. (Today)

Tomorrow is a **rest day**. Next week is **automation**: command-line apps, working with files in bulk, scheduling jobs, scraping the web responsibly, and an automation project.